In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

notebook_file = globals().get("__vsc_ipynb_file__")
NOTEBOOK_DIR = Path(notebook_file).resolve().parent if notebook_file else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))

import evaluation

DATE = "AsOfDate"

## load_data

In [ ]:
# load prices
prices = pl.read_csv(NOTEBOOK_DIR / "example_prices.csv", try_parse_dates=True).rename({"dates": DATE})
instruments = [c for c in prices.columns if c != DATE]
ret = prices.select(DATE, pl.col(instruments).diff())

In [ ]:
fig, ax = plt.subplots()
for c in instruments:
    ax.plot(prices[DATE].to_numpy(), prices[c].to_numpy(), label=c)
ax.legend()
ax.grid()

## enter your own trading strategy here

In [ ]:
def trend_model_vectorized_risk_adjusted(
    ret: pl.DataFrame, trend_window: int = 50, vol_window: int = 100, risk_window: int = 100
) -> pl.DataFrame:
    cols = [c for c in ret.columns if c != DATE]
    # calculate vol
    vol = ret.select(DATE, pl.col(cols).pow(2).rolling_sum(vol_window, min_samples=vol_window // 2).sqrt())
    # calculate signal
    signal = ret.select(DATE, pl.col(cols).rolling_sum(trend_window).sign())
    # scale with inverse vol to get pos
    positions = signal.select(DATE, *[(pl.col(c) / vol[c]).alias(c) for c in cols])
    # calculate model risk
    gross = positions.select(DATE, *[(pl.col(c).shift(2) * ret[c]).alias(c) for c in cols])
    risk = gross.filter(~pl.all_horizontal(pl.col(cols).is_null())).select(
        DATE, pl.sum_horizontal(pl.col(cols)).rolling_std(risk_window, min_samples=20).alias("model_risk")
    )
    # adjust with model risk
    aligned = positions.join(risk, on=DATE, how="left", maintain_order="left")
    return aligned.select(DATE, *[(pl.col(c) / pl.col("model_risk")).fill_nan(None).alias(c) for c in cols])

position = trend_model_vectorized_risk_adjusted(ret)

## evaluate results

In [ ]:
evaluation.plot_key_figures(position, prices)

## save your positions to a file

In [ ]:
position.write_csv(NOTEBOOK_DIR / "results.csv")